In [1]:
import sys
from pathlib import Path
base_path = Path('../../..')
sys.path.insert(0, str(base_path))

In [2]:
import warnings
warnings.filterwarnings(action='ignore')


import numpy as np
import torch
import random

from erasers import *
from configs import *

from data.utils import load_data, load_biasbios_texts, load_biasbios_texts_cf

# For embedding inversion
import vec2text
from transformers import AutoTokenizer

# For counterfactual generation evaluation
from sacrebleu.metrics import BLEU
from rouge import Rouge
import bert_score
import nltk
from nltk.tokenize import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer

# For optimal transport calculations to evaluate the Linear OT steering method
import ot

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 42

# control of randomness
if seed is not None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

dataset_name = 'biasbios-v2t'

In [4]:
# Load the dataset
# ------------------------------------------------------------------
x, z, y, _, _, _, x_test, z_test, y_test, z_values, y_values = load_data(dataset_name=dataset_name, base_path=base_path)

# print the shape of the data
print(f"Train data shape: {x.shape}, {z.shape}, {y.shape}")
print(f"Test data shape: {x_test.shape}, {z_test.shape}, {y_test.shape}")
print(f"Unique concept labels (z): {z_values} ({len(z_values)} unique values)")
print(f"Unique downstream task labels (y): {y_values} ({len(y_values)} unique values)")

Train data shape: (255710, 768), (255710,), (255710,)
Test data shape: (98344, 768), (98344,), (98344,)
Unique concept labels (z): ['f' 'm'] (2 unique values)
Unique downstream task labels (y): ['accountant' 'architect' 'attorney' 'chiropractor' 'comedian' 'composer'
 'dentist' 'dietitian' 'dj' 'filmmaker' 'interior_designer' 'journalist'
 'model' 'nurse' 'painter' 'paralegal' 'pastor' 'personal_trainer'
 'photographer' 'physician' 'poet' 'professor' 'psychologist' 'rapper'
 'software_engineer' 'surgeon' 'teacher' 'yoga_teacher'] (28 unique values)


In [5]:
# Load the counterfactual data and identify failed augmentations (the _gt suffix is for ground truth)
x_cf_gt, z_cf_gt, _, _, _, _, x_test_cf_gt, z_test_cf_gt, _, z_values_cf_gt, _ = load_data(dataset_name=dataset_name + '-cf', base_path=base_path)
failed_cf_train = z_cf_gt == 2
print(f"Number of failed counterfactual augmentations in the training set: {failed_cf_train.sum()} out of {len(z_cf_gt)} samples ({100*failed_cf_train.mean():.2f}%)")
failed_cf_test = z_test_cf_gt == 2
print(f"Number of failed counterfactual augmentations in the test set: {failed_cf_test.sum()} out of {len(z_test_cf_gt)} samples ({100*failed_cf_test.mean():.2f}%)")

Number of failed counterfactual augmentations in the training set: 2340 out of 255710 samples (0.92%)
Number of failed counterfactual augmentations in the test set: 1090 out of 98344 samples (1.11%)


In [6]:
# Let's filter out the samples for which the counterfactual augmentation failed
x_cf_gt = x_cf_gt[~failed_cf_train]
z_cf_gt = z_cf_gt[~failed_cf_train]


x_test_cf_gt = x_test_cf_gt[~failed_cf_test]
z_test_cf_gt = z_test_cf_gt[~failed_cf_test]


x = x[~failed_cf_train]
z = z[~failed_cf_train]
y = y[~failed_cf_train]

x_test = x_test[~failed_cf_test]
z_test = z_test[~failed_cf_test]
y_test = y_test[~failed_cf_test]

In [7]:
# Load test texts and their counterfactual ground truth versions
_,_,test_texts = load_biasbios_texts(base_path=base_path)
_,_,test_texts_cf_gt = load_biasbios_texts_cf(base_path=base_path)

test_texts_cf_gt = [text for text, failed in zip(test_texts_cf_gt, failed_cf_test) if not failed]
test_texts = [text for text, failed in zip(test_texts, failed_cf_test) if not failed]

In [8]:
# Display a few examples of original and counterfactual texts with their corresponding gender labels (z=0 for female, z=1 for male)
for i  in range(5):
    print(f"Original text (z={z_test[i]}): {test_texts[i]['hard_text_untokenized']}")
    print(f"Counterfactual text (z={z_test_cf_gt[i]}): {test_texts_cf_gt[i]['text_gender_reversed']}")
    print("-"*50)

Original text (z=1): Mr. Bezinque helps clients regain control of their lives by exploring solutions and taking actions to achieve their needs, both in the Collaborative Law process and in court. Collaborative Divorce Law
Counterfactual text (z=0): Ms . Bezinque helps clients regain control of their lives by exploring solutions and taking actions to achieve their needs, both in the Collaborative Law process and in court . Collaborative Divorce Law
--------------------------------------------------
Original text (z=0): She has a Ph.D. from the University of Maryland. Martin, whose teaching specialty is comparative politics and international relations with an emphasis on globalization and the developing world, previously taught at Georgetown High School and at Coastal. Her book, "The Globalization of Contentious Politics: The Amazonian Indigenous Rights Movement," was published earlier this year.
Counterfactual text (z=1): He has a Ph.D. from the University of Maryland . Martin, whose te

In [9]:
# Let's write a function to save a list of texts in a file with one text per line
def save_texts_to_file(texts, filename, title_line, output_path=base_path / "outputs/biasbios_cf_texts"):
    with open(output_path / filename, "w") as f:
        f.write(f"{title_line}\n")
        for text in texts:
            f.write(f"{text}\n")

# Gender Erasure and Counterfactual Generation of Representations

In [10]:
# fit the MUtE* model
erasure_model = MUtE(nsteps=4, optimal_erasure=True)

with torch.no_grad():
    x_erased = erasure_model.fit_transform(x, z)
    # Generate the counterfactual representations for the test set
    x_test_erased = erasure_model.transform(x_test, z_test)

    x_cf = erasure_model.inverse_transform(x_erased, z_cf_gt)
    x_test_cf = erasure_model.inverse_transform(x_test_erased, z_test_cf_gt)

# Free up memory
del erasure_model
torch.cuda.empty_cache()


Iteration 4/4 complete.: 100%|██████████| 4/4 [00:03<00:00,  1.05it/s]


In [11]:
# Do quick evaluation of the counterfactual representations
l2_norm = np.linalg.norm((x_test_cf_gt - x_test_cf), axis=1).mean()
print(f"Average L2 norm of the error of regression models on the test set: {l2_norm}")
cosine_similarity = np.mean([np.dot(x_test_cf_gt[i], x_test_cf[i]) / (np.linalg.norm(x_test_cf_gt[i]) * np.linalg.norm(x_test_cf[i])) for i in range(len(x_test)) if not failed_cf_test[i]])
print(f"Average cosine similarity between original and regressed test samples: {cosine_similarity}")

Average L2 norm of the error of regression models on the test set: 0.23867154121398926
Average cosine similarity between original and regressed test samples: 0.9362908005714417


# Embedding inversion using vec2text

In [12]:
# filter texts in the train set for which the number of tokens is less than 32, since the inversion model was trained on texts with a maximum length of 32 tokens
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/gtr-t5-base")
test_texts_small_ids = [i for i in range(len(test_texts)) if len(tokenizer.tokenize(test_texts[i]['hard_text'])) <= 32]

test_texts_small = [test_texts[i] for i in test_texts_small_ids]
test_texts_small = [t['hard_text_untokenized'] for t in test_texts_small]
test_texts_cf_gt_small = [test_texts_cf_gt[i] for i in test_texts_small_ids]
test_texts_cf_gt_small = [t['text_gender_reversed'] for t in test_texts_cf_gt_small]

x_test_small = x_test[test_texts_small_ids]
z_test_small = z_test[test_texts_small_ids]
x_test_cf_small = x_test_cf[test_texts_small_ids]

print("Number of small test texts (<= 32 tokens):", len(test_texts_small))

save_texts_to_file(test_texts_small, title_line="Small test texts (<= 32 tokens)", filename="small_test_texts.txt")
save_texts_to_file(test_texts_cf_gt_small, title_line="Counterfactuals of small test texts (<= 32 tokens)", filename="small_test_texts_cf.txt")


Number of small test texts (<= 32 tokens): 628


In [15]:
corrector = vec2text.load_pretrained_corrector("gtr-base")

def predict_text_from_embeddings(embeddings_tensor: torch.Tensor, batch=64) -> list[str]:
    """
    Takes a BERT embeddings tensor of shape (n, 768) and outputs the predicted texts.
    """
    # Ensure proper shape
    assert embeddings_tensor.ndim == 2 and embeddings_tensor.size(1) == 768, "Input must be of shape (n, 768)"
    for i in range(0, embeddings_tensor.size(0), batch):
        batch_embeddings = embeddings_tensor[i:i+batch]
        # Run the inversion algorithm to get string counterfactuals
        predicted_texts = vec2text.invert_embeddings(
            embeddings=batch_embeddings,
            corrector=corrector,
            num_steps=20,  # 20 steps is the standard iterative generation default
            sequence_beam_width=2,  # Use beam search with a beam size of 2 for better generation quality
        )
        if i == 0:
            all_predicted_texts = predicted_texts
        else:
            all_predicted_texts.extend(predicted_texts)
    return all_predicted_texts

Loading checkpoint shards: 100%|██████████| 6/6 [00:00<00:00, 114.86it/s]


In [16]:
n_samples = 5
x_test_sampled = x_test_cf_small[:n_samples]  # Take the first n_samples from the counterfactual test set
sampled_texts = test_texts_small[:n_samples]  # Corresponding original texts for reference

# Generate predicted texts from the sampled embeddings
texts_pred = predict_text_from_embeddings(
    torch.from_numpy(x_test_sampled).float().to(device)
    )
# Print the comparison of original and predicted texts
print(f"\n{n_samples} examples of original vs. predicted texts:")
for original, predicted in zip(sampled_texts, texts_pred):
    print(f"- Original: {original}")
    print(f"- Predicted: {predicted}\n")


5 examples of original vs. predicted texts:
- Original: Her research interests include international communication, media coverage of Muslims, Islamic material culture and postcolonial criticism. She can be contacted at
- Predicted: His research interests include material media coverage of Muslim culture, postcolonial criticism and muslim and international communication. These can be contacted at 

- Original: Her favorite memory from school was in 6th grade when she and her classmates had the opportunity to throw water balloons at the principal during field day.
- Predicted: His favorite memory of his principal day at school was in 6th grade when he got the chance to throw water balloons at the field of surgery.

- Original: He understands that many clients are not familiar with the legal system, and will work to make their experience as comfortable as possible. Firm Information
- Predicted: She understands that many clients will feel comfortable, as they do not have any knowledge of

# Evaluation of the generated counterfactuals

In [17]:
bleu_metric = BLEU()
rouge_metric = Rouge()

In [ ]:
# A function that takes as input a text and a gender label. 
# The function calculates the number of gender indicators (from 2 predefined lists, one for each gender) in the text.
# Then it outputs the proportion of gender indicators in the text that correspond to the target gender label.

# Ensure the required NLTK tokenizer models are available
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    nltk.download('punkt', quiet=True)

def custom_tokenize(text: str) -> list:
    """
    Tokenizes text using NLTK's word_tokenize, but enforces a specific 
    custom constraint: abbreviations like "Mr." are forcibly split into 
    the title and the period (e.g., ["Mr", "."]).

    Args:
        text (str): The input text string to tokenize.

    Returns:
        list: A list of string tokens with the applied constraints.
    """
    # Standard NLTK tokenization
    raw_tokens = word_tokenize(text)
    
    processed_tokens = []
    for token in raw_tokens:
        # Enforce the constraint: if the token is exactly "Mr.", split it.
        if token == "Mr.":
            processed_tokens.extend(["Mr", "."])
        elif token == "Ms.":
            processed_tokens.extend(["Ms", "."])
        elif token == "Mrs.":
            processed_tokens.extend(["Mrs", "."])
        elif token == "M.":
            processed_tokens.extend(["Mr", "."])
        elif token == "_.":
            processed_tokens.extend(["_", "."])
        else:
            processed_tokens.append(token)     
    return processed_tokens

# count the number of male and female indicators in the text
def count_gender_indicators(text: str) -> tuple[int, int]:
    male_indicators = ["he", "him", "his", "mr", "m", "himself"]
    female_indicators = ["she", "her", "hers", "ms", "mrs", "herself"]
    tokens = custom_tokenize(text) 
    # print(tokens)
    male_count = sum(1 for token in tokens if token.lower() in male_indicators)
    female_count = sum(1 for token in tokens if token.lower() in female_indicators)
    return male_count, female_count 

def evaluate_text(text, target_gender):
    male_count, female_count = count_gender_indicators(text)
    if target_gender == "male":
        return male_count / (male_count + female_count) if (male_count + female_count) > 0 else 0
    elif target_gender == "female":
        return female_count / (male_count + female_count) if (male_count + female_count) > 0 else 0
    else:
        raise ValueError("Invalid target gender. Choose 'male' or 'female'.")

# let's apply this function to a set of texts along with their target gender
def corpus_gender_evaluation(texts, targets):
    results = []
    for text, target in zip(texts, targets):
        if target == 0: 
            ratio = evaluate_text(text, "female")
        elif target == 1:
            ratio = evaluate_text(text, "male")
        results.append(ratio)
    return np.mean(results)



In [19]:
text_example = "Mr. Smith is here. He will meet you at noon."
print("Original text:" , text_example)
tokens = custom_tokenize(text_example)
print("Tokenized text:" , tokens)
print("Gender indicators count:" , count_gender_indicators(text_example))


male_ratio = evaluate_text(text_example, "male")
female_ratio = evaluate_text(text_example, "female")
print("Ratios of gender indicators in the text:")
print(f"Male: {male_ratio:.2f}")
print(f"Female: {female_ratio:.2f}")

corpus_result = corpus_gender_evaluation(
    ["Mr. Smith is here. He will meet you at noon.", "Ms. Johnson is at home. He will come to you at lunch."], 
    [1, 1] 
    ) # Expected result is (1.0 + 0.5) / 2 = 0.75 
print(f"Corpus gender evaluation result: {corpus_result:.2f}")

Original text: Mr. Smith is here. He will meet you at noon.
Tokenized text: ['Mr', '.', 'Smith', 'is', 'here', '.', 'He', 'will', 'meet', 'you', 'at', 'noon', '.']
Gender indicators count: (2, 0)
Ratios of gender indicators in the text:
Male: 1.00
Female: 0.00
Corpus gender evaluation result: 0.75


## Evaluation for counterfactuals generated using MUtE* 

In [20]:
texts_cf_preds = predict_text_from_embeddings(
    torch.from_numpy(x_test_cf_small).float().to(device)
    )

print(bleu_metric.corpus_score(texts_cf_preds, [test_texts_cf_gt_small]))
print(rouge_metric.get_scores(texts_cf_preds, test_texts_cf_gt_small, avg=True))
print(f"Gender substitution rate: {corpus_gender_evaluation([t for t in texts_cf_preds], 1-z_test_small):.2f}")
P, R, F1 = bert_score.score(texts_cf_preds, test_texts_cf_gt_small, lang='en')
print(f"Bert-Score: Precision: {P.mean().item():.4f}, Recall: {R.mean().item():.4f}, F1: {F1.mean().item():.4f}")

save_texts_to_file(texts_cf_preds, title_line="########## Predicted counterfactuals of small test texts (<= 32 tokens) using MUTE", filename="MUtE_cf.txt")

BLEU = 32.93 67.8/38.5/25.3/17.8 (BP = 1.000 ratio = 1.045 hyp_len = 17016 ref_len = 16280)
{'rouge-1': {'r': 0.6788710261684339, 'p': 0.6587656300203582, 'f': 0.6672852440979178}, 'rouge-2': {'r': 0.38532789195125283, 'p': 0.367761453586541, 'f': 0.3754874975893087}, 'rouge-l': {'r': 0.5794593662404366, 'p': 0.5620514813225489, 'f': 0.5694581774461617}}
Gender substitution rate: 0.85
Bert-Score: Precision: 0.9301, Recall: 0.9348, F1: 0.9324


## Evaluation for counterfactuals generated using Mean-Diff translation

In [22]:
mean_train_0, mean_train_1 = x[z==0].mean(axis=0), x[z==1].mean(axis=0) # 0 is 'f', 1 is 'm'
translation = mean_train_1 - mean_train_0 # f -> m translation vector in the original space
x_test_cf_small_md = np.zeros_like(x_test_small)
x_test_cf_small_md[z_test_small==0] = x_test_small[z_test_small==0] + translation 
x_test_cf_small_md[z_test_small==1] = x_test_small[z_test_small==1] - translation

texts_cf_preds_md = predict_text_from_embeddings(
    torch.from_numpy(x_test_cf_small_md).float().to(device)
    )

print(bleu_metric.corpus_score(texts_cf_preds_md, [test_texts_cf_gt_small]))
print(rouge_metric.get_scores(texts_cf_preds_md, test_texts_cf_gt_small, avg=True))
print(f"Gender substitution rate: {corpus_gender_evaluation([t for t in texts_cf_preds_md], 1-z_test_small):.2f}")
P, R, F1 = bert_score.score(texts_cf_preds_md, test_texts_cf_gt_small, lang='en')
print(f"Bert-Score: Precision: {P.mean().item():.4f}, Recall: {R.mean().item():.4f}, F1: {F1.mean().item():.4f}")

save_texts_to_file(texts_cf_preds_md, title_line="########## Predicted counterfactuals of small test texts (<= 32 tokens) using Mean Difference", filename="MeanDiff_cf.txt")

BLEU = 61.00 84.7/65.2/54.1/46.4 (BP = 1.000 ratio = 1.046 hyp_len = 17032 ref_len = 16280)
{'rouge-1': {'r': 0.8609382562903141, 'p': 0.826074959706904, 'f': 0.8424295259662072}, 'rouge-2': {'r': 0.6629193561959222, 'p': 0.6256872809155687, 'f': 0.6431174498963534}, 'rouge-l': {'r': 0.7935529670191012, 'p': 0.7615880873772151, 'f': 0.7765931306150459}}
Gender substitution rate: 0.46
Bert-Score: Precision: 0.9691, Recall: 0.9712, F1: 0.9701


## Evaluation for counterfactuals generated using the Linear Optimal Transport steering method

In [24]:
# Let's also compare to the linear OT steering method, which consists in learning a linear mapping from the original representations to the counterfactual ones, using optimal transport. We can use the POT library to do this.
lin_ot_01 = ot.da.LinearTransport(reg=1e-2).fit(Xs=x[z==0], Xt=x[z==1])
lin_ot_10 = ot.da.LinearTransport(reg=1e-2).fit(Xs=x[z==1], Xt=x[z==0])
x_test_cf_small_ot = np.zeros_like(x_test_small)
x_test_cf_small_ot[z_test_small==0] = lin_ot_01.transform(x_test_small[z_test_small==0])
x_test_cf_small_ot[z_test_small==1] = lin_ot_10.transform(x_test_small[z_test_small==1])

texts_cf_preds_ot = predict_text_from_embeddings(
    torch.from_numpy(x_test_cf_small_ot).float().to(device)
    )

print(bleu_metric.corpus_score(texts_cf_preds_ot, [test_texts_cf_gt_small]))
print(rouge_metric.get_scores(texts_cf_preds_ot, test_texts_cf_gt_small, avg=True))
print(f"Gender substitution rate: {corpus_gender_evaluation([t for t in texts_cf_preds_ot], 1-z_test_small):.2f}")
P, R, F1 = bert_score.score(texts_cf_preds_ot, test_texts_cf_gt_small, lang='en')
print(f"Bert-Score: Precision: {P.mean().item():.4f}, Recall: {R.mean().item():.4f}, F1: {F1.mean().item():.4f}")

save_texts_to_file(texts_cf_preds_ot, title_line="########## Predicted counterfactuals of small test texts (<= 32 tokens) using Linear Optimal Transport Steering", filename="LinearOT_cf.txt")

BLEU = 60.45 85.1/65.3/53.2/45.2 (BP = 1.000 ratio = 1.050 hyp_len = 17096 ref_len = 16280)
{'rouge-1': {'r': 0.8722654612822605, 'p': 0.8358848077796985, 'f': 0.8529936963533333}, 'rouge-2': {'r': 0.665379498541252, 'p': 0.6273821889918817, 'f': 0.6452038796039167}, 'rouge-l': {'r': 0.7980613325354156, 'p': 0.7651151347939701, 'f': 0.7806241308316471}}
Gender substitution rate: 0.65
Bert-Score: Precision: 0.9685, Recall: 0.9710, F1: 0.9698


## Evaluation for counterfactuals generated using a Linear Surrogate to MUtE*

In [26]:

from sklearn.linear_model import LinearRegression

reg01 = LinearRegression().fit(
    np.concatenate([x[z==0], x_cf[z_cf_gt==0]], axis=0),
    np.concatenate([x_cf[z_cf_gt==1], x[z==1]], axis=0)
)

reg10 = LinearRegression().fit(
    np.concatenate([x[z==1], x_cf[z_cf_gt==1]], axis=0),
    np.concatenate([x_cf[z_cf_gt==0], x[z==0]], axis=0)
)

x_test_cf_small_linreg = np.zeros_like(x_test_small)
x_test_cf_small_linreg[z_test_small==0] = reg01.predict(x_test_small[z_test_small==0])
x_test_cf_small_linreg[z_test_small==1] = reg10.predict(x_test_small[z_test_small==1])

texts_cf_preds_linreg = predict_text_from_embeddings(
    torch.from_numpy(x_test_cf_small_linreg).float().to(device)
    )

print(bleu_metric.corpus_score(texts_cf_preds_linreg, [test_texts_cf_gt_small]))
print(rouge_metric.get_scores(texts_cf_preds_linreg, test_texts_cf_gt_small, avg=True))
print(f"Gender substitution rate: {corpus_gender_evaluation([t for t in texts_cf_preds_linreg], 1-z_test_small):.2f}")
P, R, F1 = bert_score.score(texts_cf_preds_linreg, test_texts_cf_gt_small, lang='en')
print(f"Bert-Score: Precision: {P.mean().item():.4f}, Recall: {R.mean().item():.4f}, F1: {F1.mean().item():.4f}")

save_texts_to_file(texts_cf_preds_linreg, title_line="########## Predicted counterfactuals of small test texts (<= 32 tokens) using Linear Regression surrogate to MUtE*", filename="MUtE_LinReg_surrogate_cf.txt")

BLEU = 31.18 67.2/37.3/23.6/16.0 (BP = 1.000 ratio = 1.048 hyp_len = 17065 ref_len = 16280)
{'rouge-1': {'r': 0.6715354247172223, 'p': 0.6484915577270911, 'f': 0.6584108028405737}, 'rouge-2': {'r': 0.373091622116417, 'p': 0.3562796216376857, 'f': 0.36366423031333395}, 'rouge-l': {'r': 0.5721449815333894, 'p': 0.5525389323816836, 'f': 0.560975293832114}}
Gender substitution rate: 0.92
Bert-Score: Precision: 0.9300, Recall: 0.9347, F1: 0.9323


## Baseline evaluation (reconstructing the original counterfactual texts from their original embeddings)

In [27]:
# baseline (reconstructing the original text from the original embeddings without any steering)
x_test_cf_gt_small = x_test_cf_gt[test_texts_small_ids]
texts_cf_preds_gt = predict_text_from_embeddings(
    torch.from_numpy(x_test_cf_gt_small).float().to(device)
)

print(bleu_metric.corpus_score(texts_cf_preds_gt, [test_texts_cf_gt_small]))
print(rouge_metric.get_scores(texts_cf_preds_gt, test_texts_cf_gt_small, avg=True))
print(f"Gender substitution rate: {corpus_gender_evaluation([t for t in texts_cf_preds_gt], 1-z_test_small):.2f}")
P, R, F1 = bert_score.score(texts_cf_preds_gt, test_texts_cf_gt_small, lang='en')
print(f"Bert-Score: Precision: {P.mean().item():.4f}, Recall: {R.mean().item():.4f}, F1: {F1.mean().item():.4f}")

save_texts_to_file(texts_cf_preds_gt, title_line="########## Predicted counterfactuals of small test texts (<= 32 tokens) using original embeddings (no steering)", filename="baseline_cf.txt")

BLEU = 78.68 93.3/81.1/73.7/68.8 (BP = 1.000 ratio = 1.014 hyp_len = 16516 ref_len = 16280)
{'rouge-1': {'r': 0.9326345445433447, 'p': 0.9076004251794454, 'f': 0.9195043155886089}, 'rouge-2': {'r': 0.8095062361316014, 'p': 0.7850310647575212, 'f': 0.7967082437321934}, 'rouge-l': {'r': 0.8905137449620081, 'p': 0.8669483936447027, 'f': 0.8781730674202245}}
Gender substitution rate: 0.98
Bert-Score: Precision: 0.9781, Recall: 0.9807, F1: 0.9794
